# Playbook 3 · The eval gates

**Stage:** two executable pass/fail stages with structured, actionable reasons.
Readiness decides whether the **run** ships; the seasonal-naive gate decides
whether the **model** ships.


> **Playbook, not walkthrough.** [`exploration.ipynb`](exploration.ipynb) is the
> narrative for a reviewer: what was built and why. These five are operational —
> one per assignment stage, each answering *what does this stage guarantee* and
> *what would it take to run it in production*. They overlap deliberately on
> evidence and not at all on purpose.


## What this stage must guarantee

1. **A verdict, not a warning.** Exit code 0 or not. A gate that only logs is
   decoration.
2. **Reasons an operator can act on** — a code, a count, and sample keys, not a
   stack trace.
3. **Readiness runs first and blocks.** Scoring a dataset you have just declared
   untrustworthy produces a number that means nothing.

## Why readiness distrusts the query it gates

The gate re-checks the cutoff predicate with its **own independent query**. A
change to `asof_join.sql` that admitted future data would have to be made
identically in two places to go unnoticed.

That duplication is deliberate and is the one place in this repository where
duplicated logic is the point.
`tests/test_gates.py::test_readiness_gate_does_not_trust_the_query_it_gates`
holds the line.

## The full reason-code inventory

**Readiness — 12 static codes, plus one `QUARANTINED_<disposition>` per
quarantine reason (4), so 16 in total.**

| Code | Class | Meaning |
| --- | --- | --- |
| `CUTOFF_VIOLATION` | **correctness** | A selected publication postdates its cutoff |
| `SOURCE_ROWS_UNACCOUNTED` | **correctness** | Ledger disagrees with the file row counts |
| `SCHEMA_DRIFT` | source shape | A header no longer matches the parser |
| `AMBIGUOUS_MODEL_IN_USE` | source shape | ERCOT flagged more than one model in use |
| `INCOMPLETE_ZONE_HOUR_COVERAGE` | completeness | Fewer zone-hours than the calendar implies |
| `MISSING_ASOF_FORECAST` | completeness | No forecast was publishable by the cutoff |
| `MISSING_ACTUAL` | completeness | No actual to score against |
| `MISSING_NAIVE_INPUT` | completeness | No seasonal-naive input was publishable |
| `CONFLICTING_ASOF_FORECAST` | ambiguity | Two values at the winning timestamp |
| `CONFLICTING_ASOF_ACTUAL` | ambiguity | ” for the actual |
| `CONFLICTING_NAIVE_INPUT` | ambiguity | ” for the naive input |
| `STALE_FORECAST_VINTAGE` | freshness | Newest publishable vintage older than tolerance |
| `QUARANTINED_PARSE_ERROR` | quarantine | Row unparseable |
| `QUARANTINED_INVALID_VALUE` | quarantine | Value outside plausible bounds |
| `QUARANTINED_KEY_CONFLICT` | quarantine | Same key, different value, same file |
| `QUARANTINED_SCHEMA_ERROR` | quarantine | Row does not fit the expected shape |

**Model gate — 4 codes:** `INCOMPLETE_SCORING`, `WAPE_ABOVE_THRESHOLD`,
`FOLD_WAPE_ABOVE_THRESHOLD`, `PEAK_HOUR_ERROR_ABOVE_THRESHOLD`.

In [1]:
from __future__ import annotations

import datetime as dt
import tempfile
import textwrap
from pathlib import Path

from forecast_spine import coverage, fixtures, gates, normalize, pipeline, seasonal_naive

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
RAW = REPO / "data" / "raw"
SQL = REPO / "sql" / "asof_join.sql"

# The assignment window. Actuals for operating day D publish on D+1, so the
# processing date is one day past the last target day.
PROCESSING_DATE = dt.date(2026, 3, 24)
WINDOW_START, WINDOW_END = dt.date(2026, 2, 22), dt.date(2026, 3, 23)
PUBLICATION_START = dt.date(2026, 2, 21)

LIVE = any(RAW.glob("load_forecast/*_csv.zip"))
raw_root = RAW if LIVE else fixtures.build("pass", Path(tempfile.mkdtemp()))
if not LIVE:
    print("LIVE VINTAGES NOT FOUND -- using synthetic fixtures.\n"
          "Structure is preserved; scale and revision behaviour are not.\n")


def build():
    """Build a throwaway warehouse. Never touches data/warehouse/.

    A scratch database keeps this notebook runnable while something else holds
    the committed one -- DuckDB is single-writer, and a SQL client with an open
    connection is enough to block it.
    """
    if LIVE:
        context = pipeline.build_context(
            PROCESSING_DATE, raw_root, window_start=WINDOW_START, window_end=WINDOW_END
        )
    else:
        context = pipeline.build_context(
            fixtures.processing_date_for("pass"), raw_root, window_days=1
        )
    con = pipeline.connect(Path(tempfile.mkdtemp()) / "playbook.duckdb")
    pipeline.load(con, context)
    pipeline.build_evaluation_dataset(con, context, SQL)
    return con, context

## Run it — both verdicts on the assignment window

In [2]:
con, context = build()

readiness = gates.data_readiness(con, context)
model_gate = gates.seasonal_naive_gate(con, context)
naive = seasonal_naive.evaluate(con, "naive")
ercot_model = seasonal_naive.evaluate(con, "ercot")


def show(result) -> None:
    print(f"[{result.status}] {result.gate}")
    for reason in result.reasons:
        print(f"  {reason.code} (count={reason.count})")
        print("   ", textwrap.fill(reason.detail, 92, subsequent_indent="    "))
        for sample in reason.sample_keys[:2]:
            print(f"      e.g. {sample}")
    if not result.reasons:
        print("  no reasons -- every check passed")


show(readiness)
print()
show(model_gate)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

[FAIL] data_readiness
  MISSING_ASOF_FORECAST (count=8)
    8 target zone-hours where no ERCOT forecast was publishable by the cutoff; these are never
    back-filled from a later vintage
      e.g. COAST 2026-02-22T06:00Z
      e.g. EAST 2026-02-22T06:00Z
  MISSING_NAIVE_INPUT (count=8)
    8 target zone-hours where no seasonal-naive input was publishable by the cutoff; these are
    never back-filled from a later vintage
      e.g. COAST 2026-03-15T07:00Z
      e.g. EAST 2026-03-15T07:00Z
  STALE_FORECAST_VINTAGE (count=16)
    16 target zone-hours whose newest publishable forecast was up to 3.5h old at its cutoff
    (limit 2.0h). NP3-565 publishes hourly, so this indicates missed acquisition rather than
    a late ERCOT publication
      e.g. COAST 2026-03-07T19:00Z 3.5
      e.g. EAST 2026-03-07T19:00Z 3.5

[FAIL] seasonal_naive
  INCOMPLETE_SCORING (count=8)
    scored 5744 of 5752 evaluation rows; the metric does not cover the dataset it reports on
  WAPE_ABOVE_THRESHOLD (count=

### Reading the three readiness failures

| Code | Rows | Class | Defect? |
| --- | --- | --- | --- |
| `MISSING_ASOF_FORECAST` | 8 | boundary | **No.** 2026-02-22 HE 1 has a cutoff of 2026-02-21 00:00; publications post at HH:30. The assignment's own publication window starts 30 minutes too late. |
| `MISSING_NAIVE_INPUT` | 8 | structural | **No.** 2026-03-15 HE 3's week-ago input is 2026-03-08 HE 3, the hour spring-forward deletes. |
| `STALE_FORECAST_VINTAGE` | 16 | freshness | **Yes.** Five publications absent on 2026-03-06 left the newest publishable vintage 2.5h and 3.5h old against a 2h limit. |

Three failures, three different causes, three different correct responses. That
is what "structured and actionable" has to mean in practice — the operator can
triage without opening the data.

## Run it — the thresholds, and the problem with them

In [3]:
t = gates.Thresholds()
rows = [
    ("pooled WAPE", naive.wape_pct, t.max_wape_pct),
    ("worst single day (fold WAPE)", naive.worst_fold_wape_pct, t.max_fold_wape_pct),
    ("worst zone-day peak-hour APE", naive.max_peak_hour_ape_pct, t.max_peak_hour_ape_pct),
]
print(f"{'measure':32s} {'observed':>10s} {'limit':>8s}   verdict")
for label, observed, limit in rows:
    print(f"{label:32s} {observed:9.2f}% {limit:7.1f}%   {'PASS' if observed <= limit else 'FAIL'}")
print()
print(f"ERCOT's own model over the same rows: {ercot_model.wape_pct:.2f}% WAPE")
print(f"seasonal-naive bias {naive.bias_mw:+,.0f} MW   p95 APE {naive.p95_ape_pct:.1f}%")

measure                            observed    limit   verdict
pooled WAPE                           8.65%     8.0%   FAIL
worst single day (fold WAPE)         15.08%     9.0%   FAIL
worst zone-day peak-hour APE         32.15%    20.0%   FAIL

ERCOT's own model over the same rows: 3.39% WAPE
seasonal-naive bias -68 MW   p95 APE 25.0%


The thresholds were fixed against the **September** window, where seasonal-naive
scored 6.35%. Against March it scores 8.65% and all three trip.

**The decision is to leave them and report the failure.** They were set before
the March data existed, which is the only property that makes a threshold mean
anything. Re-fitting now would be fitting the gate to the answer.

In production this is a process problem, not a number problem, and the fix is
structural — see below.

## How a bad forecast still passes

| | |
| --- | --- |
| **The hole** | Pooled WAPE is dominated by the large zones and by the many cheap overnight hours. A forecast that is excellent at 03:00 in a small zone and catastrophic at the `NORTH_CENTRAL` peak clears an 8% pooled threshold comfortably. |
| **What closes it** | A second, independent limit on peak-hour error, **per zone-day** rather than pooled. `tests/test_gates.py::test_a_good_average_does_not_excuse_a_bad_peak_hour` corrupts one zone-day peak, confirms pooled WAPE still passes, and shows this guardrail is what refuses the release. |
| **What stays open** | A **uniformly, mildly biased** forecast. Bias cancels in an absolute-error metric and nothing here gates on it. The compensating signal is the signed `bias_mw` already computed per fold — a run of same-signed daily bias is the observable symptom. |

## In production

**Thresholds belong in versioned configuration, not in a dataclass**, and each
one carries provenance: the value, the date it was set, the window it was fitted
on, and who approved it. That metadata is what turns "8.0%" from a magic number
into a reviewable decision — and it is exactly what was missing when a September
threshold met March.

**Seasonality-aware limits.** One fixed number across a year is wrong for load.
A rolling per-calendar-month baseline — say the trailing three years of the same
month, re-fitted on a schedule and reviewed, never silently — is the actual fix.
Named here as work not done rather than quietly implemented.

**Gate as a deployment step.** Exit code drives the release. Readiness failure
must block *publication of the evaluation*, not merely annotate it, or the
number leaks into a dashboard and becomes the thing people cite.

**Routing by class, not by code.** Sixteen codes is too many to page on
individually:

| Class | Severity | Route |
| --- | --- | --- |
| correctness (`CUTOFF_VIOLATION`, `SOURCE_ROWS_UNACCOUNTED`) | page immediately | on-call data engineer |
| ambiguity (`CONFLICTING_*`) | block, ticket | data engineer + source owner |
| source shape (`SCHEMA_DRIFT`, `AMBIGUOUS_MODEL_IN_USE`) | block, ticket | pipeline owner |
| freshness (`STALE_FORECAST_VINTAGE`) | block, ticket | acquisition owner — see Playbook 0 |
| completeness (`MISSING_*`) | block, triage | may be structural; needs judgment, not a rule |
| model (`*_ABOVE_THRESHOLD`) | block release, no page | forecasting owner, next business day |

**Weekly drift monitor** *(bonus in the brief, not built)*: track pooled WAPE,
signed bias and peak-hour APE per zone over a trailing window. The symptom that
trips it is a run of same-signed daily bias — the gap the peak-hour guardrail
does not close. Alerts the forecasting owner, not on-call.

## Runbook

| Verdict | Meaning | Action |
| --- | --- | --- |
| Both pass | Release approved | Publish. Record `run_id` with the artifact. |
| Readiness fails, correctness class | **Incident** | Halt. Quarantine downstream artifacts. Do not re-run hoping it passes. |
| Readiness fails, completeness class | Triage | Structural (boundary, DST) → report and exclude explicitly. Retrieval → Playbook 0, then re-run. |
| Readiness fails, freshness class | Investigate | Almost always our acquisition, not ERCOT's lateness. |
| Model gate fails | Model not fit to ship | Report. **Do not adjust the threshold in the same change that observed the failure.** |
| Both pass but downstream disputes the number | Suspect a silent leak | Compare as-of WAPE against the latest-vintage WAPE. A large gap is expected; no gap is the alarm. |

## Where this lives

`src/forecast_spine/gates.py` · `seasonal_naive.py` · `tests/test_gates.py` ·
`forecast-spine demo` (five fixture scenarios, each verdict asserted)